# Integrar módulos con el runner encapsulado

## 1. Crear únicamente tres archivos

| Archivo nuevo | Responsabilidad |
|---|---|
| `frontend/src/models/orders.py` | Contratos de entrada y salida. |
| `frontend/src/services/orders_service.py` | Peticiones y validación del dominio. |
| `frontend/src/ui/views/orders.py` | Campos, presentación y permisos del menú. |

No editar Services, SistemaGestion, login ni AsyncRunner. Los servicios se descubren por el sufijo `_service.py`, y las vistas por sus metadatos. Reiniciar la app después de añadir archivos. Los módulos se importan al arrancar: no colocar scripts con efectos secundarios en estas carpetas.

Arrancar desde la raíz con `python frontend/src/app/main.py`. El punto de entrada incorpora `frontend/src` al recorrido de imports, por eso se usan `models`, `services` y `ui` sin el prefijo `frontend.src`.

El ejemplo propone GET `/ordenes/` y POST `/ordenes/`. Estos endpoints aún deben implementarse en el backend. El servidor valida equipo, asigna estado inicial y autoriza cada operación; revisar requisitos, SQL y diagrama antes de adoptar este contrato.

## 2. Modelo: contenido de models/orders.py

```python
from pydantic import Field
from models.base import Contrato, Mensaje


class OrdenCrear(Contrato):
    """equipo_id: identificador de un equipo existente."""
    equipo_id: int = Field(gt=0)


class Orden(Contrato):
    """Registro público del servidor."""
    id: int = Field(gt=0)
    equipo_id: int = Field(gt=0)
    estado: str


class OrdenRegistrada(Mensaje):
    """Confirmación: mensaje heredado y orden creada."""
    orden: Orden
```

`Field` aquí es Pydantic; `gt=0` exige un entero positivo. `Contrato` valida tipos estrictos sin convertir texto a entero, ignora extras y hace inmutables los datos. `Mensaje` exige `mensaje` no vacío. Separar entrada y salida permite que el servidor asigne ID y estado.

## 3. Servicio: contenido de services/orders_service.py

```python
from services.api_client import ModuleService
from models.orders import OrdenCrear, Orden, OrdenRegistrada


class OrdersService(ModuleService):
    """Operaciones de órdenes con el transporte compartido."""
    service_name = 'orders'

    async def listar(self) -> list[Orden]:
        return await self._list('/ordenes/', Orden)

    async def crear(self, equipo_id) -> OrdenRegistrada:
        entrada = self._parse(OrdenCrear, {'equipo_id': equipo_id}, entrada=True)
        data = await self.api.request(
            'POST', '/ordenes/', json=entrada.model_dump(mode='json'))
        return self._parse(OrdenRegistrada, data)
```

| Elemento | Parámetros y motivo |
|---|---|
| `service_name` | Clave pública `services.orders`; si se omite, se usa el nombre del archivo sin `_service.py`. |
| Constructor heredado `__init__(api)` | Recibe el ApiClient existente; no crear HTTP ni runner nuevos. |
| `listar()` | Retorna lista de modelos, requerida por la consulta común de la vista. |
| `crear(equipo_id)` | Recibe entero positivo ya convertido por la vista. |
| `_list(path, model)` | Ruta relativa de GET y contrato de cada registro; exige lista JSON. |
| `_parse(model, data, entrada=True)` | Contrato y datos; identifica errores de formulario. |
| `_parse(OrdenRegistrada, data)` | Valida confirmación antes de mostrar éxito. |
| `request(method, path, json=...)` | Método, ruta y cuerpo; ApiClient aplica JWT y traduce errores HTTP. |

El archivo debe terminar en `_service.py` y definir una clase que herede ModuleService. Services importa esa clase y la construye con la API compartida; no registra otra vez clases importadas desde otro archivo. La clave debe ser única, identificador Python válido y no reservada. Claves duplicadas provocan error al iniciar.

Usar `params` para filtros y `json` para cuerpos JSON. `mode='json'` serializa tipos como fechas cuando se añadan. Dejar que ApiError llegue al callback visual: devolver una lista vacía ante un error ocultaría el fallo como ausencia de datos.

## 4. Vista: contenido de ui/views/orders.py

```python
from models.orders import Orden
from services.api_client import ApiError
from ui.components import CollectionView, Field


class OrdersView(CollectionView):
    """Listado y alta; la base administra solicitudes y formularios."""
    menu_label = 'Órdenes'
    allowed_roles = ('admin', 'recepcion')
    menu_order = 25
    service_name = 'orders'
    title = 'Órdenes de servicio'
    columns = ('ID', 'ID Equipo', 'Estado')
    fields = (Field('equipo_id', 'ID del equipo'),)

    async def save(self, values):
        try:
            equipo_id = int(values['equipo_id'])
        except ValueError:
            raise ApiError('El ID del equipo debe ser un entero.') from None
        return await self.service.crear(equipo_id=equipo_id)

    def row(self, record: Orden):
        return (record.id, record.equipo_id, record.estado)
```

| Miembro | Propósito |
|---|---|
| `menu_label` | Texto del botón; sin etiqueta no se incorpora al menú. |
| `allowed_roles` | Roles API que pueden ver la pantalla; el backend también verifica permisos. |
| `menu_order` | Orden ascendente; empate por etiqueta; valor predeterminado 100. |
| `service_name` | Conecta la vista con `services.orders`. |
| `title` | Encabezado y título del formulario. |
| `columns` | Encabezados, en el orden de `row`. |
| `fields` | Claves y etiquetas del formulario. Aquí Field es de UI, no Pydantic. |
| `save(values)` | Diccionario capturado por el formulario; convierte ID explícitamente. |
| `row(record)` | Modelo leído; devuelve valores de presentación, sin consultar HTTP. |

No hace falta escribir `fetch`: la base llama a `self.service.listar()`. Su `save` predeterminado llama a `self.service.crear(**values)`; aquí lo sobrescribimos porque Entry entrega texto y el contrato exige entero. Sobrescribir fetch solo si se necesitan filtros o consultas propias.

El constructor heredado recibe `master` (contenedor Tk), `services` (servicios compartidos), `runner` (infraestructura interna) y `role` (rol de sesión). No definirlo si no se agregan controles; si se redefine, conservar la firma y llamar a super. La base carga al construirse: preparar previamente atributos utilizados por fetch.

`Field(name, label, required=True, secret=False, choices=())`: name identifica la clave de values, label el texto, required exige presencia, secret oculta y conserva espacios, choices genera selector. `EquipmentView` muestra un selector que traduce etiquetas a IDs.

La base administra carga, formulario, errores, Actualizar y recarga después del alta. La confirmación debe ser compatible con Mensaje. El folio mostrado por la base sigue siendo específico de equipos. No duplicar submit ni crear tareas en fetch/save.

## 5. Operación personalizada sin conocer el runner

Para una pantalla distinta de listado/alta, heredar AsyncFrame y construir controles. Usar este patrón dentro de un método normal de la vista:

```python
try:
    equipo_id = int(self.entry_equipo.get())
except ValueError:
    self.status.configure(text='El ID debe ser un entero.')
    return
self.button.configure(state='disabled')
self.request(self.services.orders.crear, equipo_id,
             on_success=self._saved, on_error=self._failed)
```

`request(operation, *args, on_success, on_error=None, **kwargs)` recibe el método async sin llamarlo, los datos y callbacks normales. La base crea la coroutine, la vincula a esta vista y programa su ejecución. `on_success(result)` recibe el resultado; `on_error(error)` recibe una excepción. Si se omite on_error, se usa `_failed`, que muestra el error en `self.status`. Si se deshabilitan botones, definir `_saved` y `_failed` para restaurarlos.

Leer widgets antes de request. Los services no acceden a Tk; los callbacks sí corren en su hilo principal. AsyncFrame cancela tareas al destruirse; un destroy propio debe llamar a super. El Future retornado solo es necesario para cancelar una lectura específica; no esperar con `.result()` en Tk ni usar asyncio.run en botones.

El hilo y la cola permanecen encapsulados. Cambiar de vista cancela sus tareas, no cierra la API compartida. Cerrar la aplicación sigue siendo responsabilidad del login. Cancelar una petición no revierte un POST ya recibido. La revisión de la cola con after no consulta Supabase; Actualizar es una nueva lectura, sin suscripción en tiempo real.

## 6. Validación y errores frecuentes

1. Reiniciar la app y comprobar el menú con cada rol.
2. Comprobar que Services expone orders y comparte api con otros servicios.
3. Usar MockTransport siguiendo el laboratorio v2: GET y POST válidos, payload, entrada inválida sin HTTP, error 403 y respuesta malformada.
4. Comprobar row con una Orden y abrir/cerrar mientras una consulta está pendiente.
5. Ejecutar `python -m unittest discover -s frontend/tests -v` y después probar contra el backend real.

Los tests de `test_module_integration.py` verifican descubrimiento sin editar contenedores, roles, cliente compartido, claves duplicadas y delegación de operaciones. No certifican los endpoints de órdenes ni la apariencia visual.

| Síntoma | Revisar |
|---|---|
| Servicio ausente | Sufijo `_service.py`, herencia y service_name. |
| Vista ausente | Archivo en ui/views, herencia AsyncFrame, etiqueta y roles. |
| Error al arrancar | Importación incorrecta o clave duplicada/reservada. |
| Error de contrato | Tipos estrictos o JSON incompatible. |
| HTTP 404 | Router de órdenes pendiente o ruta distinta. |
| HTTP 403 | Permisos del backend. |
| Interfaz congelada | Trabajo bloqueante en Tk; evitar `.result()` y asyncio.run. |

Los bloques Python muestran contenido para los archivos indicados. No se ejecutan al correr el notebook; la siguiente celda solo valida su sintaxis.

In [ ]:
from pathlib import Path
import ast
import re
import nbformat

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'frontend/src/app/main.py').is_file()), None)
if raiz is None:
    raise RuntimeError('Abre este notebook dentro del repositorio Fixify.')
notebook = nbformat.read(raiz / 'docs/GUIA_INTEGRACION_MODULOS_FRONTEND.ipynb', 4)
ejemplos = []
for celda in notebook.cells:
    if celda.cell_type == 'markdown':
        ejemplos.extend(re.findall(r'```python\n(.*?)```', celda.source, re.S))
for indice, ejemplo in enumerate(ejemplos, 1):
    ast.parse(ejemplo, filename=f'ejemplo_{indice}')
print(f'{len(ejemplos)} bloques Python con sintaxis válida. No se ejecutó la app.')
